# 02 · Baselines and validation (Experiment A)

* **A2 parity** (CPU, minutes): local DTI vs an independent implementation (and the official scorer if you place it in `data/external/`).
* **A1 + A4** (GPU, ~30 min): trains the missing pseudo-discovery runs of the gated dual encoder with the *Phase-4* code and protocol, then scores G1, RS05, CNN-A, CNN-B, reduced-modality CNN and the gated encoder under **one** protocol (same folds, splits, budgets, policies) and writes the competition-alignment scoreboard.
* **A3 designs** (CPU): system-k-fold, short-fault (<2 km) and leave-one-region-out designs saved as standard split files.

Every stage is restartable: re-run a cell after a disconnect and completed work is skipped.

In [ ]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'mybenkhadda/GEMS---drivendata'
COMMIT = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://github.com/{REPO}.git'   # token-free remote: a token never goes into the URL / .git/config
# private fork only: GITHUB_TOKEN (Colab Secret) is sent as an HTTP header through env vars, per command
genv = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()} if token else None
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True, env=genv)
subprocess.run(['git', '-C', '/content/GEMS', 'remote', 'set-url', 'origin', url], check=True)  # drop an old embedded token
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True, env=genv)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
colab.stage_core(); colab.stage_legacy();

In [ ]:
from gems_bench.experiment import Experiment
C = '/content/GEMS/configs/experiments/'
a2 = Experiment.from_yaml(C + 'a2_parity.yaml')
a2.run_all()
display(a2.rd.load_table('parity_cases'))
display(a2.rd.load_table('conventions'))

In [ ]:
# A1 + A4 (GPU): complete P4_B_gated pseudo-discovery, then the unified scoreboard
a1 = Experiment.from_yaml(C + 'a1_zoo_discovery.yaml')
a1.run_all()
board = a1.rd.load_table('scoreboard')
board[['model', 'geo_dti_area', 'geo_dti_raw', 'geo_fold_min', 'geo_fold_std', 'disc_dti_excl', 'disc_dti_retained_fp',
       'short_recall_w', 'public_lb']]

In [ ]:
from IPython.display import Image, Markdown
display(Markdown(a1.rd.path.joinpath('reports', 'experiment_summary.md').read_text()))
display(Image(str(a1.rd.figures / 'experiment_dashboard.png')))

In [ ]:
# A3: stronger spatial designs (saved under data/processed/splits/ and pushed to Drive)
a3 = Experiment.from_yaml(C + 'a3_splits.yaml')
a3.run_all()
a3.rd.load_table('design_summary')